# FSBL example (wide + caustic-crossing)

Finite-source binary-lens photometry and astrometry from `bagle.model_jax`.

Two geometries:
1. **Wide** — large impact parameter; trajectory misses the caustics.
2. **Narrow / caustic-crossing** — small impact parameter on a close binary.

For each case we plot:
- unlensed source and lens positions (with caustics),
- lensed image positions and magnifications,
- photometric light curve,
- lensed astrometric trajectory (centroid).

**Units (phot+astrom):** `sep` in mas; `radiusS` in arcsec;
`rho = radiusS * 1e3 / thetaE_amp`.

In [ ]:
%matplotlib inline
from bagle import model_jax as model
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def build_fsbl_psbl(beta, sep, alpha, radiusS, n_outline=100,
                    mLp=0.5, mLs=0.5, t0=57000.0, dL=4000.0, dS=8000.0,
                    muS_E=5.0):
    """
    Build matching FSBL / PSBL PhotAstrom_noPar_Param1 models.

    Parameters
    ----------
    beta, sep, alpha, radiusS, n_outline
        Geometry and source size (mas, mas, deg, arcsec).
    mLp, mLs, t0, dL, dS, muS_E
        Shared event parameters.

    Returns
    -------
    fsbl, psbl : model instances
    """
    xS0_E, xS0_N = [0.0], [0.0]
    b_sff, mag_src, dmag = [1.0], [18.0], [20.0]

    fsbl = model.FSBL_PhotAstrom_noPar_Param1(
        mLp, mLs, t0, radiusS, xS0_E, xS0_N, beta,
        0.0, 0.0, muS_E, 0.0, dL, dS, sep, alpha,
        b_sff, mag_src, dmag, n_outline=n_outline)
    psbl = model.PSBL_PhotAstrom_noPar_Param1(
        mLp, mLs, t0, xS0_E, xS0_N, beta,
        0.0, 0.0, muS_E, 0.0, dL, dS, sep, alpha,
        b_sff, mag_src, dmag)

    return fsbl, psbl


def plot_unlensed_geometry(fsbl, t, title):
    """Unlensed source track, lens tracks, and caustics at t0."""
    xS = np.asarray(fsbl.get_astrometry_unlensed(t)) * 1e3
    rL = np.asarray(fsbl.get_resolved_lens_astrometry(t)) * 1e3
    caus = np.asarray(fsbl.get_caustics(np.array([fsbl.t0])))[0]  # (N, 4) complex

    fig, ax = plt.subplots(figsize=(6, 5))
    # Caustics in Einstein radii -> mas.
    for k in range(caus.shape[1]):
        zc = caus[:, k] * fsbl.thetaE_amp
        ax.plot(np.real(zc), np.imag(zc), 'C3.', ms=1, alpha=0.35,
                label='caustics' if k == 0 else None)

    ax.plot(xS[:, 0], xS[:, 1], 'C0-', lw=1.5, label='source (unlensed)')
    ax.plot(xS[0, 0], xS[0, 1], 'C0o', ms=5)
    ax.plot(rL[0, :, 0], rL[0, :, 1], 'k-', label='lens 1')
    ax.plot(rL[1, :, 0], rL[1, :, 1], color='0.45', label='lens 2')
    ax.plot(rL[0, 0, 0], rL[0, 0, 1], 'k*', ms=10)
    ax.plot(rL[1, 0, 0], rL[1, 0, 1], '*', color='0.45', ms=10)

    ax.set_xlabel(r'$\Delta\alpha^*$ (mas)')
    ax.set_ylabel(r'$\Delta\delta$ (mas)')
    ax.set_aspect('equal')
    ax.invert_xaxis()
    ax.legend(fontsize=8, loc='best')
    ax.set_title(title)
    fig.tight_layout()
    plt.show()

    return None


def plot_lensed_images(fsbl, psbl, t, title):
    """Point-source image tracks (PSBL) colored by magnification."""
    # Image positions / amps from the matching PSBL (source-center map).
    img, amp = psbl.get_all_arrays(t)
    img = np.asarray(img)   # (N_t, 5) complex, Einstein radii
    amp = np.asarray(amp)   # (N_t, 5)

    # Absolute sky positions: lens COM + image * thetaE.
    xL_com = np.asarray(fsbl.get_lens_astrometry(t)) * 1e3  # mas
    thE = fsbl.thetaE_amp

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Left: image tracks on the sky.
    ax = axes[0]
    for i in range(img.shape[1]):
        xi = xL_com[:, 0] + np.real(img[:, i]) * thE
        yi = xL_com[:, 1] + np.imag(img[:, i]) * thE
        ok = np.isfinite(xi) & np.isfinite(yi) & np.isfinite(amp[:, i])
        sc = ax.scatter(xi[ok], yi[ok], c=amp[ok, i], s=8, cmap='viridis',
                        label=f'image {i}')
    rL = np.asarray(fsbl.get_resolved_lens_astrometry(t)) * 1e3
    ax.plot(rL[0, :, 0], rL[0, :, 1], 'k*', ms=8, label='lens 1')
    ax.plot(rL[1, :, 0], rL[1, :, 1], '*', color='0.45', ms=8, label='lens 2')
    fig.colorbar(sc, ax=ax, label='|A| (image)')
    ax.set_xlabel(r'$\Delta\alpha^*$ (mas)')
    ax.set_ylabel(r'$\Delta\delta$ (mas)')
    ax.set_aspect('equal')
    ax.invert_xaxis()
    ax.set_title('Lensed image positions')
    ax.legend(fontsize=7, loc='best')

    # Right: per-image magnification vs time.
    ax = axes[1]
    for i in range(amp.shape[1]):
        ai = np.abs(amp[:, i])
        ok = np.isfinite(ai)
        ax.plot(t[ok] - fsbl.t0, ai[ok], label=f'image {i}')
    A_tot = np.asarray(fsbl.get_amplification(t))
    ax.plot(t - fsbl.t0, A_tot, 'k-', lw=2, label='FSBL total')
    ax.set_xlabel(r'$t - t_0$ (days)')
    ax.set_ylabel('Amplification')
    ax.set_yscale('log')
    ax.legend(fontsize=7, loc='best')
    ax.set_title('Image and total magnification')

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

    return None


def plot_photometry(fsbl, psbl, t, title):
    """Photometric light curve: FSBL vs PSBL."""
    mag_f = np.asarray(fsbl.get_photometry(t))
    mag_p = np.asarray(psbl.get_photometry(t))
    amp_f = np.asarray(fsbl.get_amplification(t))
    amp_p = np.asarray(psbl.get_amplification(t))

    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                             gridspec_kw=dict(height_ratios=[3, 1]))
    axes[0].plot(t - fsbl.t0, mag_f, 'k-', label='FSBL')
    axes[0].plot(t - fsbl.t0, mag_p, 'C0--', label='PSBL')
    axes[0].invert_yaxis()
    axes[0].set_ylabel('mag')
    axes[0].legend()
    ax2 = axes[0].twinx()
    ax2.plot(t - fsbl.t0, amp_f, '0.5:', alpha=0.8)
    ax2.set_ylabel('A (FSBL)', color='0.4')
    axes[0].set_title(title)

    axes[1].plot(t - fsbl.t0, mag_f - mag_p, 'k-')
    axes[1].axhline(0, color='0.5', lw=0.8)
    axes[1].set_xlabel(r'$t - t_0$ (days)')
    axes[1].set_ylabel('FSBL − PSBL')
    fig.tight_layout()
    plt.show()

    print(f"A_max FSBL={amp_f.max():.3f}, PSBL={amp_p.max():.3f}")

    return None


def plot_lensed_astrometry(fsbl, psbl, t, title):
    """Lensed centroid trajectory and centroid shift."""
    xF = np.asarray(fsbl.get_astrometry(t)) * 1e3
    xP = np.asarray(psbl.get_astrometry(t)) * 1e3
    xU = np.asarray(fsbl.get_astrometry_unlensed(t)) * 1e3
    shF = np.asarray(fsbl.get_centroid_shift(t)) * 1e3
    shP = np.asarray(psbl.get_centroid_shift(t)) * 1e3

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    ax = axes[0]
    ax.plot(xU[:, 0], xU[:, 1], 'C0--', alpha=0.7, label='unlensed source')
    ax.plot(xF[:, 0], xF[:, 1], 'k-', label='FSBL centroid')
    ax.plot(xP[:, 0], xP[:, 1], 'C1:', label='PSBL centroid')
    ax.plot(xF[0, 0], xF[0, 1], 'ko', ms=4)
    ax.set_xlabel(r'$\Delta\alpha^*$ (mas)')
    ax.set_ylabel(r'$\Delta\delta$ (mas)')
    ax.set_aspect('equal')
    ax.invert_xaxis()
    ax.legend(fontsize=8)
    ax.set_title('Lensed centroid trajectory')

    ax = axes[1]
    ax.plot(t - fsbl.t0, shF[:, 0], 'k-', label='FSBL E')
    ax.plot(t - fsbl.t0, shF[:, 1], 'k--', label='FSBL N')
    ax.plot(t - fsbl.t0, shP[:, 0], 'C1-', alpha=0.7, label='PSBL E')
    ax.plot(t - fsbl.t0, shP[:, 1], 'C1--', alpha=0.7, label='PSBL N')
    ax.set_xlabel(r'$t - t_0$ (days)')
    ax.set_ylabel('centroid shift (mas)')
    ax.legend(fontsize=8)
    ax.set_title('Centroid shift')

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

    return None


def run_suite(fsbl, psbl, t, label):
    """Run the four standard plot panels for one geometry."""
    rho = fsbl.radiusS * 1e3 / fsbl.thetaE_amp
    print(f"{label}: thetaE={fsbl.thetaE_amp:.3f} mas, "
          f"u0={fsbl.u0_amp:.3f}, rho={rho:.4f}, tE={fsbl.tE:.1f} d")

    plot_unlensed_geometry(fsbl, t, f'{label}: unlensed source and lenses')
    plot_lensed_images(fsbl, psbl, t, f'{label}: lensed images')
    plot_photometry(fsbl, psbl, t, f'{label}: photometry')
    plot_lensed_astrometry(fsbl, psbl, t, f'{label}: lensed astrometry')

    return None

## 1. Wide (non-caustic)

Large impact parameter so the source misses the caustics. Finite-source
effects are mild; FSBL and PSBL stay close.

| Param | Value | Units |
|---|---|---|
| $m_{Lp}, m_{Ls}$ | 0.5, 0.5 | $M_\odot$ |
| $t_0$ | 57000 | MJD |
| $\beta$ | 2.0 | mas |
| $d_L, d_S$ | 4000, 8000 | pc |
| $sep$ | 1.0 | mas |
| $\alpha$ | 45 | deg |
| $radiusS$ | $2\times10^{-4}$ | arcsec |
| $n_{outline}$ | 100 | |

In [ ]:
fsbl_w, psbl_w = build_fsbl_psbl(
    beta=2.0, sep=1.0, alpha=45.0, radiusS=2e-4, n_outline=100)
t_w = np.linspace(fsbl_w.t0 - 1.5 * fsbl_w.tE,
                  fsbl_w.t0 + 1.5 * fsbl_w.tE, 120)
run_suite(fsbl_w, psbl_w, t_w, 'Wide')

## 2. Narrow (caustic-crossing)

Close binary with a small impact parameter so the source crosses the
caustic. Finite-source smoothing of the peaks is clear versus PSBL.

| Param | Value | Units |
|---|---|---|
| $m_{Lp}, m_{Ls}$ | 0.5, 0.5 | $M_\odot$ |
| $t_0$ | 57000 | MJD |
| $\beta$ | 0.02 | mas |
| $d_L, d_S$ | 4000, 8000 | pc |
| $sep$ | 0.8 | mas |
| $\alpha$ | 0 | deg |
| $radiusS$ | $1\times10^{-4}$ | arcsec |
| $n_{outline}$ | 120 | |

In [ ]:
fsbl_c, psbl_c = build_fsbl_psbl(
    beta=0.02, sep=0.8, alpha=0.0, radiusS=1e-4, n_outline=120)
# Denser sampling near peak for the caustic features.
t_c = np.linspace(fsbl_c.t0 - 1.2 * fsbl_c.tE,
                  fsbl_c.t0 + 1.2 * fsbl_c.tE, 160)
run_suite(fsbl_c, psbl_c, t_c, 'Caustic-crossing')